# Training

学習条件を上流から確認し、事前チェック後にsmoke testまたは本学習を明示的に実行し、保存済み結果を可視化します。

このNotebookはYAMLを書き換えません。設定を変更した場合は先頭から再実行してください。


## 1. 使用する設定

通常はこのパスだけ確認します。別実験のYAMLを使う場合のみ変更してください。


In [ ]:
from pathlib import Path
import csv, subprocess, sys
import matplotlib.pyplot as plt
import yaml
from IPython.display import Image, Markdown, display

def find_project_root(start):
    start = Path(start).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "configs").is_dir() and (candidate / "performance").is_dir():
            return candidate
    raise FileNotFoundError("FNOV2リポジトリ内からNotebookを開いてください。")

def resolve(value, config_path):
    path = Path(value)
    return path.resolve() if path.is_absolute() else (config_path.parent / path).resolve()

def status(ok, label, detail=""):
    mark = "✅" if ok else "❌"
    print(f"{mark} {label}" + (f": {detail}" if detail else ""))
    return ok

ROOT = find_project_root(Path.cwd())
print("project:", ROOT)
#################################
EXPERIMENT_ID = "ex008_baseline"
#################################
print("experiment:", EXPERIMENT_ID)
TRAIN_YAML = ROOT / "configs/train.yaml"
train_cfg = yaml.safe_load(TRAIN_YAML.read_text(encoding="utf-8"))
model_path = resolve(train_cfg["training"]["model_config"], TRAIN_YAML)
model_cfg = yaml.safe_load(model_path.read_text(encoding="utf-8"))
loader_path = resolve(model_cfg["model"]["dataloader_config"], model_path)
loader_cfg = yaml.safe_load(loader_path.read_text(encoding="utf-8"))
dataset_path = resolve(loader_cfg["dataset_config"], loader_path)
dataset_cfg = yaml.safe_load(dataset_path.read_text(encoding="utf-8"))
output_dir = resolve(train_cfg["output"]["directory"], TRAIN_YAML)


## 2. 設定の可視化


In [ ]:
summary = {
    "train config": str(TRAIN_YAML),
    "model": model_cfg["model"]["name"],
    "modes / hidden / layers": [model_cfg["model"]["n_modes"], model_cfg["model"]["hidden_channels"], model_cfg["model"]["n_layers"]],
    "train Re": loader_cfg["split"]["train_re"],
    "validation Re": loader_cfg["split"]["val_re"],
    "batch size": loader_cfg["loader"]["batch_size"],
    "epochs / device": [train_cfg["training"]["epochs"], train_cfg["training"]["device"]],
    "optimizer / learning rate": [train_cfg["optimizer"]["type"], train_cfg["optimizer"]["learning_rate"]],
    "scheduler": train_cfg["scheduler"],
    "loss": train_cfg["loss"],
    "normalization": dataset_cfg["normalization"],
    "W&B": train_cfg.get("wandb", {}),
    "output": str(output_dir),
}
display(Markdown("### 重要設定"))
print(yaml.safe_dump(summary, allow_unicode=True, sort_keys=False))
display(Markdown("### train.yaml 全文"))
print(TRAIN_YAML.read_text(encoding="utf-8"))


## 3. 実行前チェック


In [ ]:
data_dir = resolve(dataset_cfg["dataset"]["data_dir"], dataset_path)
pair_files = sorted(data_dir.glob(dataset_cfg["dataset"]["file_pattern"]))
train_re = set(map(float, loader_cfg["split"]["train_re"]))
val_re = set(map(float, loader_cfg["split"]["val_re"]))
checks = [
    status(TRAIN_YAML.is_file(), "train config", TRAIN_YAML),
    status(model_path.is_file(), "model config", model_path),
    status(loader_path.is_file(), "dataloader config", loader_path),
    status(dataset_path.is_file(), "dataset config", dataset_path),
    status(data_dir.is_dir(), "training data directory", data_dir),
    status(bool(pair_files), "pair files", f"{len(pair_files)} files"),
    status(train_re.isdisjoint(val_re), "train/validation Re separation"),
    status(output_dir.name in ({"ex008"} if EXPERIMENT_ID == "ex008_baseline" else {EXPERIMENT_ID}), "experiment training output", output_dir),
]
resume = train_cfg["training"].get("resume_checkpoint")
if resume:
    checks.append(status(resolve(resume, TRAIN_YAML).is_file(), "resume checkpoint", resolve(resume, TRAIN_YAML)))
PRECHECK_OK = all(checks)
print("\n実行可能" if PRECHECK_OK else "\n設定またはデータを修正してから実行してください。")


## 4. Smoke test

1 train batchと1 validation batchだけを実行し、結果ファイルは保存しません。


In [ ]:
RUN_SMOKE_TEST = True
if RUN_SMOKE_TEST:
    if not PRECHECK_OK: raise RuntimeError("事前チェックが失敗しています。")
    subprocess.run([sys.executable, str(ROOT/"src/train.py"), "--config", str(TRAIN_YAML), "--smoke-test"], cwd=ROOT, check=True)
else:
    print("未実行。確認後にRUN_SMOKE_TEST=Trueへ変更してください。")


## 5. 本学習

既存の出力先を使用します。output.directoryとresume_checkpointを必ず確認してから有効化してください。


In [ ]:
RUN_TRAINING = True
if RUN_TRAINING:
    if not PRECHECK_OK: raise RuntimeError("事前チェックが失敗しています。")
    subprocess.run([sys.executable, str(ROOT/"src/train.py"), "--config", str(TRAIN_YAML)], cwd=ROOT, check=True)
else:
    print("未実行。確認後にRUN_TRAINING=Trueへ変更してください。")


## 6. 保存済み学習結果


In [2]:
history_path = output_dir / "history.csv"
if not history_path.exists():
    print("学習履歴がありません:", history_path)
else:
    with history_path.open(encoding="utf-8", newline="") as f: rows=list(csv.DictReader(f))
    epoch=[int(r["epoch"]) for r in rows]
    fig,axes=plt.subplots(1,2,figsize=(13,4.5))
    axes[0].plot(epoch,[float(r["train_loss"]) for r in rows],label="train")
    axes[0].plot(epoch,[float(r["val_loss"]) for r in rows],label="validation")
    axes[1].plot(epoch,[float(r["train_relative_l2"]) for r in rows],label="train")
    axes[1].plot(epoch,[float(r["val_relative_l2"]) for r in rows],label="validation")
    axes[0].set(xlabel="epoch",ylabel="MSE loss",title="Loss history")
    axes[1].set(xlabel="epoch",ylabel="Relative L2",title="Relative L2 history")
    for ax in axes: ax.grid(alpha=.3); ax.legend()
    plt.tight_layout(); plt.show()
    best=min(rows,key=lambda r:float(r["val_relative_l2"]))
    print("best validation Relative L2:",best["val_relative_l2"],"at epoch",best["epoch"])
for name in ("best.pt","last.pt","history.csv","history.json"):
    status((output_dir/name).exists(),name,output_dir/name)


NameError: name 'output_dir' is not defined